# H&M Transaction Data: Product Recommendations 01
## Pre-process data

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

import os
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder

# Define paths
base_path = "../data/"

# Initialize summary report
summary = {
    'articles': {},
    'customers': {},
    'transactions': {}
}

print("=" * 60)
print("DATA CLEANING REPORT - H&M Datasets")
print("=" * 60)
print(f"Cleaning started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)


DATA CLEANING REPORT - H&M Datasets
Cleaning started at: 2026-02-28 19:33:29


## Data cleaning

In [9]:
# ============================================================
# 1. CLEAN ARTICLES DATASET
# ============================================================
print("\n[1/3] CLEANING ARTICLES DATASET")
print("-" * 40)

articles = pd.read_csv(base_path + "articles_hm.csv")
summary['articles']['original_rows'] = len(articles)
summary['articles']['original_cols'] = len(articles.columns)

print(f"Original shape: {articles.shape}")

# Check for duplicates
duplicates = articles.duplicated(subset=['article_id']).sum()
summary['articles']['duplicates_removed'] = duplicates
if duplicates > 0:
    articles = articles.drop_duplicates(subset=['article_id'], keep='first')
    print(f"Removed {duplicates} duplicate article_id rows")

# Check for missing values
missing_before = articles.isnull().sum().sum()
summary['articles']['missing_values_before'] = missing_before

# Handle missing values
# For categorical columns, fill with 'Unknown'
categorical_cols = articles.select_dtypes(include=['object']).columns
for col in categorical_cols:
    missing_count = articles[col].isnull().sum()
    if missing_count > 0:
        articles[col] = articles[col].fillna('Unknown')
        print(f"  Filled {missing_count} missing values in '{col}' with 'Unknown'")

# For numeric columns, fill with median
numeric_cols = articles.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    missing_count = articles[col].isnull().sum()
    if missing_count > 0:
        median_val = articles[col].median()
        articles[col] = articles[col].fillna(median_val)
        print(f"  Filled {missing_count} missing values in '{col}' with median ({median_val})")

summary['articles']['missing_values_after'] = articles.isnull().sum().sum()

# Ensure article_id is string and properly formatted
articles['article_id'] = articles['article_id'].astype(str).str.zfill(10)
articles['product_code'] = articles['product_code'].astype(str).str.zfill(7)

# Strip whitespace from string columns
for col in categorical_cols:
    if col in articles.columns:
        articles[col] = articles[col].str.strip()

summary['articles']['final_rows'] = len(articles)
print(f"Final shape: {articles.shape}")

# Save cleaned articles
articles.to_csv(base_path + "articles_hm_cleaned.csv", index=False)
print("Saved: articles_hm_cleaned.csv")


[1/3] CLEANING ARTICLES DATASET
----------------------------------------
Original shape: (105542, 25)
  Filled 416 missing values in 'detail_desc' with 'Unknown'
Final shape: (105542, 25)
Saved: articles_hm_cleaned.csv


In [10]:
# ============================================================
# 2. CLEAN CUSTOMERS DATASET
# ============================================================
print("\n[2/3] CLEANING CUSTOMERS DATASET")
print("-" * 40)

customers = pd.read_csv(base_path + "customer_hm.csv")
summary['customers']['original_rows'] = len(customers)
summary['customers']['original_cols'] = len(customers.columns)

print(f"Original shape: {customers.shape}")

# Check for duplicates
duplicates = customers.duplicated(subset=['customer_id']).sum()
summary['customers']['duplicates_removed'] = duplicates
if duplicates > 0:
    customers = customers.drop_duplicates(subset=['customer_id'], keep='first')
    print(f"Removed {duplicates} duplicate customer_id rows")

# Check for missing values
missing_before = customers.isnull().sum().sum()
summary['customers']['missing_values_before'] = missing_before

# Handle missing values for each column appropriately
# FN and Active - binary columns, fill with mode (0)
for col in ['FN', 'Active']:
    missing_count = customers[col].isnull().sum()
    if missing_count > 0:
        customers[col] = customers[col].fillna(0).astype(int)
        print(f"  Filled {missing_count} missing values in '{col}' with 0")

# club_member_status - fill with 'Unknown'
if customers['club_member_status'].isnull().sum() > 0:
    missing_count = customers['club_member_status'].isnull().sum()
    customers['club_member_status'] = customers['club_member_status'].fillna('Unknown')
    print(f"  Filled {missing_count} missing values in 'club_member_status' with 'Unknown'")

# fashion_news_frequency - fill with 'NONE'
if customers['fashion_news_frequency'].isnull().sum() > 0:
    missing_count = customers['fashion_news_frequency'].isnull().sum()
    customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE')
    print(f"  Filled {missing_count} missing values in 'fashion_news_frequency' with 'NONE'")

# age - fill with median, handle outliers
if customers['age'].isnull().sum() > 0:
    missing_count = customers['age'].isnull().sum()
    median_age = customers['age'].median()
    customers['age'] = customers['age'].fillna(median_age)
    print(f"  Filled {missing_count} missing values in 'age' with median ({median_age})")

# Handle age outliers (cap at reasonable bounds: 10-100)
age_outliers = ((customers['age'] < 10) | (customers['age'] > 100)).sum()
if age_outliers > 0:
    customers['age'] = customers['age'].clip(lower=10, upper=100)
    print(f"  Capped {age_outliers} age outliers to range [10, 100]")
    summary['customers']['age_outliers_fixed'] = age_outliers

summary['customers']['missing_values_after'] = customers.isnull().sum().sum()

# Standardize categorical values
customers['club_member_status'] = customers['club_member_status'].str.upper().str.strip()
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].str.upper().str.strip()

# Ensure proper data types
customers['FN'] = customers['FN'].astype(int)
customers['Active'] = customers['Active'].astype(int)
customers['age'] = customers['age'].astype(int)

summary['customers']['final_rows'] = len(customers)
print(f"Final shape: {customers.shape}")

# Save cleaned customers
customers.to_csv(base_path + "customer_hm_cleaned.csv", index=False)
print("Saved: customer_hm_cleaned.csv")


[2/3] CLEANING CUSTOMERS DATASET
----------------------------------------
Original shape: (1048575, 6)
  Filled 1 missing values in 'fashion_news_frequency' with 'NONE'
Final shape: (1048575, 6)
Saved: customer_hm_cleaned.csv


In [11]:
# ============================================================
# 3. CLEAN TRANSACTIONS DATASET
# ============================================================
print("\n[3/3] CLEANING TRANSACTIONS DATASET")
print("-" * 40)

transactions = pd.read_csv(base_path + "transactions_hm.csv")
summary['transactions']['original_rows'] = len(transactions)
summary['transactions']['original_cols'] = len(transactions.columns)

print(f"Original shape: {transactions.shape}")

# Check for duplicates (exact duplicates across all columns)
duplicates = transactions.duplicated().sum()
summary['transactions']['duplicates_removed'] = duplicates
if duplicates > 0:
    transactions = transactions.drop_duplicates(keep='first')
    print(f"Removed {duplicates} exact duplicate rows")

# Check for missing values
missing_before = transactions.isnull().sum().sum()
summary['transactions']['missing_values_before'] = missing_before

# Handle missing values
for col in transactions.columns:
    missing_count = transactions[col].isnull().sum()
    if missing_count > 0:
        print(f"  Found {missing_count} missing values in '{col}'")
        if col == 't_dat':
            transactions = transactions.dropna(subset=['t_dat'])
            print(f"    Dropped rows with missing dates")
        elif col == 'customer_id':
            transactions = transactions.dropna(subset=['customer_id'])
            print(f"    Dropped rows with missing customer_id")
        elif col == 'article_id':
            transactions = transactions.dropna(subset=['article_id'])
            print(f"    Dropped rows with missing article_id")
        elif col == 'price':
            median_price = transactions['price'].median()
            transactions[col] = transactions[col].fillna(median_price)
            print(f"    Filled with median price ({median_price})")
        elif col == 'sales_channel_id':
            mode_channel = transactions['sales_channel_id'].mode()[0]
            transactions[col] = transactions[col].fillna(mode_channel)
            print(f"    Filled with mode ({mode_channel})")

summary['transactions']['missing_values_after'] = transactions.isnull().sum().sum()

# Convert date column to datetime
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'], errors='coerce')

# Remove any rows where date conversion failed
invalid_dates = transactions['t_dat'].isnull().sum()
if invalid_dates > 0:
    transactions = transactions.dropna(subset=['t_dat'])
    print(f"Removed {invalid_dates} rows with invalid dates")
    summary['transactions']['invalid_dates_removed'] = invalid_dates

# Ensure article_id is properly formatted
transactions['article_id'] = transactions['article_id'].astype(str).str.zfill(10)

# Handle price outliers (remove negative or extremely high prices)
price_outliers = ((transactions['price'] < 0) | (transactions['price'] > 1)).sum()
if price_outliers > 0:
    transactions = transactions[(transactions['price'] >= 0) & (transactions['price'] <= 1)]
    print(f"Removed {price_outliers} rows with invalid prices (outside 0-1 range)")
    summary['transactions']['price_outliers_removed'] = price_outliers

# Ensure proper data types
transactions['sales_channel_id'] = transactions['sales_channel_id'].astype(int)
transactions['price'] = transactions['price'].astype(float)

summary['transactions']['final_rows'] = len(transactions)
print(f"Final shape: {transactions.shape}")

# Save cleaned transactions
transactions.to_csv(base_path + "transactions_hm_cleaned.csv", index=False)
print("Saved: transactions_hm_cleaned.csv")


[3/3] CLEANING TRANSACTIONS DATASET
----------------------------------------
Original shape: (1048575, 5)
Removed 8474 exact duplicate rows
Final shape: (1040101, 5)
Saved: transactions_hm_cleaned.csv


In [12]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

for dataset, stats in summary.items():
    print(f"\n{dataset.upper()}:")
    print(f"  Original rows: {stats.get('original_rows', 'N/A')}")
    print(f"  Final rows: {stats.get('final_rows', 'N/A')}")
    print(f"  Duplicates removed: {stats.get('duplicates_removed', 0)}")
    print(f"  Missing values (before): {stats.get('missing_values_before', 0)}")
    print(f"  Missing values (after): {stats.get('missing_values_after', 0)}")

print("\n" + "=" * 60)
print("CLEANING COMPLETED SUCCESSFULLY!")
print("=" * 60)
print("\nCleaned files saved:")
print("  - articles_hm_cleaned.csv")


CLEANING SUMMARY

ARTICLES:
  Original rows: 105542
  Final rows: 105542
  Duplicates removed: 0
  Missing values (before): 416
  Missing values (after): 0

CUSTOMERS:
  Original rows: 1048575
  Final rows: 1048575
  Duplicates removed: 0
  Missing values (before): 1
  Missing values (after): 0

TRANSACTIONS:
  Original rows: 1048575
  Final rows: 1040101
  Duplicates removed: 8474
  Missing values (before): 0
  Missing values (after): 0

CLEANING COMPLETED SUCCESSFULLY!

Cleaned files saved:
  - articles_hm_cleaned.csv


In [14]:
# ============================================================
# Feature Engineering and EDA will be performed in the next steps using the cleaned datasets.
# ============================================================
# quick check
print("articles_clean_df:", articles_clean_df.shape)
print("customers_clean_df:", customers_clean_df.shape)
print("transactions_clean_df:", transactions_clean_df.shape)


articles_clean_df: (105542, 25)
customers_clean_df: (1048575, 6)
transactions_clean_df: (1040101, 5)


## Data loading

In [3]:
data_path = Path("../data")
customers = pd.read_csv(data_path / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(data_path / 'transactions_hm_cleaned.csv')
articles = pd.read_csv(data_path / 'articles_hm_cleaned.csv')

In [4]:
print("Cleaned Rows of Data:")
print(f"Articles: {len(articles):,}")
print(f"Customers: {len(customers):,}")
print(f"Transactions: {len(transactions):,}")

Cleaned Rows of Data:
Articles: 105,542
Customers: 1,048,575
Transactions: 1,040,101


In [15]:
# Sample data for faster execution
TRANSACTIONS_SAMPLE_SIZE = 500000
transactions_sample = transactions.sample(n=TRANSACTIONS_SAMPLE_SIZE, random_state=67)

CUSTOMER_SAMPLE_SIZE = 500000
# sample_customers = transactions_sample['customer_id'].unique()
customers_sample = customers.sample(n=CUSTOMER_SAMPLE_SIZE, random_state=67)

transactions_df = transactions_sample
customers_df = customers_sample
articles_df = articles

In [16]:
print(f"Customers: using {len(customers_df):,} out of {len(customers):,} available")
print(f"Transactions: using {len(transactions_df):,} out of {len(transactions):,} available")
print(f"Articles: using {len(articles_df):,} out of {len(articles):,} available")

Customers: using 500,000 out of 1,048,575 available
Transactions: using 500,000 out of 1,040,101 available
Articles: using 105,542 out of 105,542 available


In [17]:
PURCHASE = "Purchase"
NO_PURCHASE = "No Purchase"
LABELS = [NO_PURCHASE, PURCHASE]

## Train, Validation Data Generation

In [18]:
def build_product_recommendation_data(as_of_date, prediction_start_date, prediction_end_date):
    print(f"Data as of date: {as_of_date}")

    cfe = CustomerFeatureEngineer(customers_df=customers_df, transactions_df=transactions_df)
    customer_features = cfe.calculate_all_features(articles_df=articles_df, as_of_date=as_of_date)
    print(f"Customer rows: {len(customer_features):,}")

    pfe = ProductFeatureEngineer(articles_df=articles_df, transactions_df=transactions_df)
    product_features = pfe.calculate_all_features(as_of_date=as_of_date)
    print(f"Product rows: {len(product_features):,}")


    print(f"Prediction period: {prediction_start_date} to {prediction_end_date}")
    builder = RecommendationTrainingBuilder(transactions_df=transactions_df,
                                                customer_features_df=customer_features,
                                                product_features_df=product_features)
    data = builder.build_dataset(prediction_start=prediction_start_date,
                                            prediction_end=prediction_end_date,
                                            negative_ratio=5,
                                            random_state=67)

    print(f"Total rows: {len(data):,}")
    print(f"- Positives: {(data['purchased'] == 1).sum():,}")
    print(f"- Negatives: {(data['purchased'] == 0).sum():,}")
    return data

In [19]:
print("=========TRAINING DATA=========")
train_as_of_date = "2019-09-30"
train_prediction_start_date = "2019-10-01"
train_prediction_end_date = "2019-10-30"
train_data =  build_product_recommendation_data(train_as_of_date, train_prediction_start_date, train_prediction_end_date)

=========TRAINING DATA=========
Data as of date: 2019-09-30
Customer rows: 252,238
Product rows: 37,553
Prediction period: 2019-10-01 to 2019-10-30
Total rows: 83,100
- Positives: 13,850
- Negatives: 69,250


In [8]:
print("\n=========VALIDATION DATA=========")
val_as_of_date = "2019-10-31"
val_prediction_start = "2019-11-01"
val_prediction_end = "2019-11-30"
val_data =  build_product_recommendation_data(val_as_of_date, val_prediction_start, val_prediction_end)


=========VALIDATION DATA=========
Data as of date: 2019-10-31
Customer rows: 268,826
Product rows: 39,630
Prediction period: 2019-11-01 to 2019-11-30
Total rows: 90,432
- Positives: 15,072
- Negatives: 75,360


In [9]:
print("\n=========TEST DATA=========")
test_as_of_date = "2019-11-30"
test_prediction_start = "2019-12-01"
test_prediction_end = "2019-12-31"
test_data =  build_product_recommendation_data(test_as_of_date, test_prediction_start, test_prediction_end)


=========TEST DATA=========
Data as of date: 2019-11-30
Customer rows: 285,551
Product rows: 41,562
Prediction period: 2019-12-01 to 2019-12-31
Total rows: 85,764
- Positives: 14,294
- Negatives: 71,470


In [10]:
feature_cols = [col for col in train_data.columns if col not in ['customer_id', 'article_id', 'purchased']]

print(f"Feature Columns:")
for col in feature_cols:
    print(f"- {col}")


Feature Columns:
- sales_last_7_days
- sales_last_30_days
- days_since_first_sale
- days_since_last_sale
- avg_price
- min_price
- max_price
- product_price_std
- customer_price_std
- num_purchases
- total_spent
- days_since_last_purchase
- avg_transaction_value
- avg_days_between_purchases
- primary_department
- primary_garment_group
- category_diversity


### Categorical features
One hot encode the garment groups. Drop primary department, too many possibilities

In [11]:
print(f"Unique departments: {train_data['primary_department'].nunique()}")
print(f"Unique garment groups: {train_data['primary_garment_group'].nunique()}")

Unique departments: 207
Unique garment groups: 21


In [12]:
train_data = train_data.drop('primary_department', axis=1)
val_data = val_data.drop('primary_department', axis=1)
test_data = test_data.drop('primary_department', axis=1)

train_data = pd.get_dummies(train_data, columns=['primary_garment_group'], prefix='garment')
val_data = pd.get_dummies(val_data, columns=['primary_garment_group'], prefix='garment')
test_data = pd.get_dummies(test_data, columns=['primary_garment_group'], prefix='garment')


all_columns = set(train_data.columns).union(val_data.columns).union(test_data.columns)
print(all_columns)

train_data = train_data.reindex(columns=all_columns, fill_value=0)
val_data = val_data.reindex(columns=all_columns, fill_value=0)
test_data = test_data.reindex(columns=all_columns, fill_value=0)

{'garment_Trousers Denim', 'garment_Skirts', 'avg_transaction_value', 'garment_Special Offers', 'max_price', 'garment_Dresses Ladies', 'garment_Knitwear', 'garment_Socks and Tights', 'garment_Unknown', 'garment_Trousers', 'total_spent', 'garment_Blouses', 'garment_Shorts', 'garment_Woven/Jersey/Knitted mix Baby', 'sales_last_7_days', 'garment_Outdoor', 'purchased', 'customer_price_std', 'garment_Dressed', 'category_diversity', 'garment_Shirts', 'avg_price', 'article_id', 'days_since_last_purchase', 'customer_id', 'garment_Dresses/Skirts girls', 'garment_Jersey Basic', 'product_price_std', 'num_purchases', 'garment_Shoes', 'days_since_last_sale', 'garment_Accessories', 'min_price', 'avg_days_between_purchases', 'garment_Under-, Nightwear', 'sales_last_30_days', 'days_since_first_sale', 'garment_Swimwear', 'garment_Jersey Fancy'}


### X, y split
Outcome variable is `purchased` and 0 or 1 feature

In [187]:
X_train = train_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_train = train_data['purchased']
print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
X_val = val_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_val = val_data['purchased']
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
X_test = test_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_test = test_data['purchased']
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(83100, 36)
y_train.shape=(83100,)
X_val.shape=(90432, 36)
y_val.shape=(90432,)
X_test.shape=(85764, 36)
y_test.shape=(85764,)


## Save Processed Data
### as pickle files

In [188]:
# Save data as parquet to avoid reprocessing

processed_data_path = data_path / 'processed' / 'product_recommendation'
with open(processed_data_path / 'X_train_base.pkl', 'wb') as f:
    pickle.dump(X_train, f)
with open(processed_data_path / 'y_train_base.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open(processed_data_path / 'X_val_base.pkl', 'wb') as f:
    pickle.dump(X_val, f)
with open(processed_data_path / 'y_val_base.pkl', 'wb') as f:
    pickle.dump(y_val, f)
with open(processed_data_path / 'X_test_base.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open(processed_data_path / 'y_test_base.pkl', 'wb') as f:
    pickle.dump(y_test, f)